preparation

- > catalog : dab_1_dev
    - Schema : default
      - Volume: health
        - dev_health.csv: Small Subset of prd, anonymized Pll, 7500 rows

- > catalog : dab_2_stage
    - Schema : default
      - Volume: health
        - stage_health.csv: Subset of prd data, anonymized Pll, 3500s rows

- > catalog : dab_3_pro
    - Schema : default
        - Volume: health
          - 2025-01-01_health.csv
          - 2025-02-01_health.csv
          - 2025-03-01_health.csv
          - CSV files are added to this cloud storage location daily

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dab_1_dev;
CREATE CATALOG IF NOT EXISTS dab_2_stage;
CREATE CATALOG IF NOT EXISTS dab_3_prod;

In [0]:
create schema if not exists dab_1_dev.default;
create schema if not exists dab_2_stage.default;
create schema if not exists dab_3_prod.default;


In [0]:
create volume if not exists dab_1_dev.default.health;
create volume if not exists dab_2_stage.default.health;
create volume if not exists dab_3_prod.default.health;

## 2 Generate Tokess

Instalar CLI

Si no se usa el CLI WEB

In [0]:
%sh
curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh


Hacerlo con Python

In [0]:
%python
import os

# Extraemos el token del scope de forma segura
os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get(scope="dab_secrets", key="dab_token")

# Configura tus datos (mejor si usas secretos como vimos antes)
os.environ['DATABRICKS_HOST'] = "https://dbc-77b2401c-4344.cloud.databricks.com/"


En el bash

In [0]:
%sh
export DATABRICKS_HOST="https://dbc-77b2401c-4344.cloud.databricks.com/"

In [0]:
%python
scope_name = "dab_secrets"
secret_key = "dab_token2"
## desabilitado mi _token = "databricks token aqui" # Tu token real

# 1. Intentamos crear el scope. Si ya existe, el error se ignora.
try:
    dbutils.secrets.get(scope=scope_name, key=secret_key)
    print(f"✅ El scope '{scope_name}' ya existe y tiene el token.")
except Exception:
    print(f"Creating scope '{scope_name}'...")
    # Nota: En algunos entornos 'create-scope' desde python requiere permisos de Admin
    # Si falla, créalo una última vez por el terminal manualmente.
    pass

# 2. Forzamos el guardado del token (por si se borró)
dbutils.secrets.put(scope=scope_name, key=secret_key, string_value=mi_token)
print("✅ Token asegurado en el Secret Scope.")

In [0]:
%python
import os

# Extraemos el token del cofre que creaste en la imagen
token = dbutils.secrets.get(scope="dab_secrets", key="dab_token")

# Seteamos las variables de entorno correctamente
os.environ['DATABRICKS_TOKEN'] = token
os.environ['DATABRICKS_HOST'] = "https://dbc-77b2401c-4344.cloud.databricks.com/" # Pon la URL que empieza por https://

print("✅ Host y Token actualizados correctamente.")

In [0]:
%python
class Config:
    def __init__(self):
        # 1. Capturamos el catálogo que inyecta el Bundle (dev, pre o pro)
        # Si corres el notebook a mano, usará 'dab_1_dev' por defecto
        self.catalog = dbutils.widgets.getArgument("catalog") if \
                       self._widget_exists("catalog") else "dab_1_dev"
        
        # 2. Definimos las rutas de las capas Medallion dinámicamente
        self.bronze = f"{self.catalog}.bronze"
        self.silver = f"{self.catalog}.silver"
        self.gold   = f"{self.catalog}.gold"
        
        # 3. Variable de usuario para evitar colisiones
        self.user_id = self.catalog.split("_") # Extrae 'dab'

    def _widget_exists(self, name):
        try:
            dbutils.widgets.get(name)
            return True
        except:
            return False

# Instanciamos el objeto 'DA' (o como quieras llamarlo)
DA = Config()

# --- PRUEBA DE FUEGO ---
print(f"🚀 Entorno activo: {DA.catalog}")
print(f"📂 Ruta Bronze:  {DA.bronze}")
print(f"📂 Ruta Silver:  {DA.silver}")

# Ahora ya puedes hacer lo mismo que el instructor:
print(DA.catalog)

In [0]:
%sh
databricks -vdatabricks
